# CMC R1 — Gemma remaining three states at 768 tokens

This notebook completes the Gemma four-state 768-token evaluation without rerunning the already completed detector-state condition.

New generation:
- llm_only
- flow
- graph

Existing detector-state 768 outputs are reused from:
`/content/drive/MyDrive/CMC_R2_gemma2_full500_768_20260919`

Expected new outputs: 3 datasets × 500 targets × 3 states = 4,500.

Pinned for comparability with the existing Gemma 768 run:
- repo commit: `98014d0aaec6ac910b4dfbc720a46385c6799f8c`
- model: `google/gemma-2-9b-it`
- model revision: `11c9b309abf73637e4b6f9a3fa1e92e615547819`
- max_new_tokens=768
- BF16, SDPA, batch size 4, do_sample=False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Clone and pin the experiment repository


In [ ]:
import os, shutil, stat, subprocess, sys, json, platform
from pathlib import Path
from google.colab import userdata

REPO_URL='https://github.com/ramita99/cmc_llm_87708.git'
PINNED_COMMIT='98014d0aaec6ac910b4dfbc720a46385c6799f8c'
REPO_DIR=Path('/content/cmc_llm_gemma_three_states_768')

github_token=userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Colab Secrets에 GITHUB_TOKEN을 추가하세요.')

askpass=Path('/content/git_askpass_cmc_gemma_768.sh')
askpass.write_text("""#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *) echo "$GITHUB_TOKEN" ;;
esac
""", encoding='utf-8')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)

env=os.environ.copy()
env['GITHUB_TOKEN']=github_token
env['GIT_ASKPASS']=str(askpass)
env['GIT_TERMINAL_PROMPT']='0'

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

try:
    subprocess.run(['git','clone',REPO_URL,str(REPO_DIR)],check=True,env=env)
    subprocess.run(['git','-C',str(REPO_DIR),'checkout',PINNED_COMMIT],check=True,env=env)
finally:
    askpass.unlink(missing_ok=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.16.1'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)

src=str(REPO_DIR/'src')
if src not in sys.path:
    sys.path.insert(0,src)

HEAD=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
assert HEAD==PINNED_COMMIT,(HEAD,PINNED_COMMIT)
print('Repo HEAD:',HEAD)


## 2. GPU and pinned Gemma model revision


In [ ]:
import torch, transformers, yaml, pandas as pd
from huggingface_hub import login, snapshot_download

if not torch.cuda.is_available():
    raise RuntimeError('GPU runtime required.')
if not torch.cuda.is_bf16_supported():
    raise RuntimeError('BF16-capable GPU required.')

props=torch.cuda.get_device_properties(0)
print(f'GPU: {props.name} | VRAM: {props.total_memory/1024**3:.1f} GiB')

hf_token=userdata.get('HF_TOKEN')
if hf_token:
    login(token=hf_token,add_to_git_credential=False)

MODEL_ID='google/gemma-2-9b-it'
MODEL_REVISION='11c9b309abf73637e4b6f9a3fa1e92e615547819'
LOCAL_MODEL=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION,token=hf_token)

print('Model revision:',MODEL_REVISION)
print('torch:',torch.__version__)
print('transformers:',transformers.__version__)


## 3. Reuse the original frozen packets and select only llm_only / flow / graph


In [ ]:
from cmc87708.protocol import assert_llm_facing_packet, stable_hash

SOURCE_ROOT=Path('/content/drive/MyDrive/CMC_R2_full_frozen_20260918')
DETECTOR_768_ROOT=Path('/content/drive/MyDrive/CMC_R2_gemma2_full500_768_20260919')
THREE_ROOT=Path('/content/drive/MyDrive/CMC_R1_gemma2_threestate_768_20260920')
COMBINED_ROOT=Path('/content/drive/MyDrive/CMC_R1_gemma2_fourstate_768_20260920')

DATASETS=['NF-UNSW-NB15-v3','NF-CSE-CIC-IDS2018-v3','NF-ToN-IoT-v3']
THREE_STATES=['llm_only','flow','graph']

for p in [SOURCE_ROOT/'packets', DETECTOR_768_ROOT/'packets',
          DETECTOR_768_ROOT/'outputs'/'proposed_source_gated']:
    if not p.exists():
        raise FileNotFoundError(p)

for sub in ['packets','manifests','results']:
    (THREE_ROOT/sub).mkdir(parents=True,exist_ok=True)

def load_jsonl(path):
    with Path(path).open('r',encoding='utf-8') as f:
        return [json.loads(x) for x in f if x.strip()]

def target_key(packet):
    t=packet['traceability']
    return (str(t.get('window_id')),str(t.get('edge_id')))

manifest=[]
for ds in DATASETS:
    all_packets=load_jsonl(SOURCE_ROOT/'packets'/f'{ds}.jsonl')
    selected=[p for p in all_packets if p.get('state') in THREE_STATES]
    counts=pd.Series([p['state'] for p in selected]).value_counts().to_dict()
    assert counts=={'llm_only':500,'flow':500,'graph':500},(ds,counts)

    detector_packets=load_jsonl(DETECTOR_768_ROOT/'packets'/f'{ds}.jsonl')
    detector_keys={target_key(p) for p in detector_packets}
    assert len(detector_keys)==500

    for state in THREE_STATES:
        state_keys={target_key(p) for p in selected if p['state']==state}
        assert state_keys==detector_keys,f'{ds} {state}: target mismatch'

    with (THREE_ROOT/'packets'/f'{ds}.jsonl').open('w',encoding='utf-8') as f:
        for p in selected:
            assert_llm_facing_packet(p,'Gemma 768 three-state packet')
            f.write(json.dumps(p,ensure_ascii=False,default=str)+'\n')
            t=p['traceability']
            manifest.append({
                'dataset':ds,'state':p['state'],
                'window_id':t.get('window_id'),'edge_id':t.get('edge_id'),
                'packet_hash':stable_hash(p)
            })

manifest=pd.DataFrame(manifest)
assert len(manifest)==4500
assert (manifest.groupby(['dataset','state']).size()==500).all()
manifest.to_csv(THREE_ROOT/'manifests'/'three_state_768_packet_manifest.csv',index=False)
display(manifest.groupby(['dataset','state']).size().rename('n').reset_index())


## 4. Write the 768-token config


In [ ]:
CONFIG={
 'protocol_version':'CMC-R1-GEMMA-THREESTATE-768-2026-09-20',
 'source_main_protocol':'CMC-R2-FROZEN-2026-09-18',
 'output_root':str(THREE_ROOT),
 'sample_seed':7,
 'targets_per_dataset':500,
 'sampling':'same_frozen_500_targets_as_main_run',
 'states':THREE_STATES,
 'methods':['proposed_source_gated'],
 'full_state_for_method_comparison':'detector',
 'model_id':MODEL_ID,
 'model_revision':MODEL_REVISION,
 'generation_dtype':'bfloat16',
 'attention_implementation':'sdpa',
 'generation_batch_size':4,
 'max_new_tokens':768,
 'do_sample':False,
 'numeric_tolerance':0.0001,
 'fallback_on_truncation':True,
 'bootstrap_replicates':10000,
 'multiple_testing_correction':'holm',
 'expected_datasets':DATASETS,
 'expected_targets_per_dataset':500,
 'expected_packet_states_per_target':3,
}

CONFIG_PATH=Path('/content/cmc_r1_gemma_three_states_768.yaml')
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG,sort_keys=False),encoding='utf-8')
(THREE_ROOT/'cmc_r1_gemma_three_states_768.yaml').write_text(
    yaml.safe_dump(CONFIG,sort_keys=False),encoding='utf-8')

envrec={
 'model_id':MODEL_ID,'model_revision':MODEL_REVISION,'repo_head':HEAD,
 'gpu':torch.cuda.get_device_name(0),'torch':torch.__version__,
 'transformers':transformers.__version__,'python':platform.python_version(),
 'max_new_tokens':768,'do_sample':False,
 'states':THREE_STATES,'expected_new_generations':4500
}
(THREE_ROOT/'execution_environment.json').write_text(
    json.dumps(envrec,indent=2),encoding='utf-8')
print(CONFIG_PATH.read_text())


## 5. Generate the remaining 4,500 Proposed outputs (resume-safe)


In [ ]:
MODEL_PATH=str(LOCAL_MODEL)
cmd=[
    sys.executable,'scripts/02_run_primary_generation.py',
    '--experiment',str(CONFIG_PATH),
    '--model',MODEL_PATH,
    '--methods','proposed_source_gated',
    '--resume'
]
print(' '.join(cmd))
subprocess.run(cmd,check=True)


## 6. Verify 4,500 outputs and recompute exact-field/audit metrics


In [ ]:
total=0
for ds in DATASETS:
    path=THREE_ROOT/'outputs'/'proposed_source_gated'/f'{ds}.jsonl'
    recs=load_jsonl(path)
    assert len(recs)==1500,(ds,len(recs))
    counts=pd.Series([r['state'] for r in recs]).value_counts().to_dict()
    assert counts=={'llm_only':500,'flow':500,'graph':500},(ds,counts)
    assert len({r['record_key'] for r in recs})==1500
    print(ds,counts)
    total+=len(recs)
assert total==4500
print('Generation complete:',total)

subprocess.run([
    sys.executable,'scripts/03_recompute_primary_metrics.py',
    '--experiment',str(CONFIG_PATH)
],check=True)

display(pd.read_csv(THREE_ROOT/'results'/'summary_metrics_audit_v2.csv'))


## 7. Merge with the existing detector-state 768 Proposed outputs


In [ ]:
for sub in ['packets','outputs/proposed_source_gated','results','manifests']:
    (COMBINED_ROOT/sub).mkdir(parents=True,exist_ok=True)

combined_manifest=[]

for ds in DATASETS:
    source_packets=load_jsonl(SOURCE_ROOT/'packets'/f'{ds}.jsonl')
    assert len(source_packets)==2000,(ds,len(source_packets))
    counts=pd.Series([p['state'] for p in source_packets]).value_counts().to_dict()
    assert counts=={'llm_only':500,'flow':500,'graph':500,'detector':500},(ds,counts)

    with (COMBINED_ROOT/'packets'/f'{ds}.jsonl').open('w',encoding='utf-8') as f:
        for p in source_packets:
            f.write(json.dumps(p,ensure_ascii=False,default=str)+'\n')

    new_three=load_jsonl(THREE_ROOT/'outputs'/'proposed_source_gated'/f'{ds}.jsonl')
    old_detector=load_jsonl(DETECTOR_768_ROOT/'outputs'/'proposed_source_gated'/f'{ds}.jsonl')

    assert len(new_three)==1500
    assert len(old_detector)==500
    assert {r['state'] for r in old_detector}=={'detector'}

    frozen_detector_hashes={stable_hash(p) for p in source_packets if p['state']=='detector'}
    detector_output_hashes={r['packet_hash'] for r in old_detector}
    assert detector_output_hashes==frozen_detector_hashes,f'{ds}: detector packet hash mismatch'

    merged=new_three+old_detector
    assert len(merged)==2000
    merged_counts=pd.Series([r['state'] for r in merged]).value_counts().to_dict()
    assert merged_counts=={'llm_only':500,'flow':500,'graph':500,'detector':500}
    assert len({r['record_key'] for r in merged})==2000

    out=COMBINED_ROOT/'outputs'/'proposed_source_gated'/f'{ds}.jsonl'
    with out.open('w',encoding='utf-8') as f:
        for r in merged:
            f.write(json.dumps(r,ensure_ascii=False,default=str)+'\n')

    for r in merged:
        p=r['packet']; t=p['traceability']
        combined_manifest.append({
            'dataset':ds,'state':r['state'],
            'window_id':t.get('window_id'),'edge_id':t.get('edge_id'),
            'packet_hash':r['packet_hash'],'output_hash':r.get('output_hash')
        })

combined_manifest=pd.DataFrame(combined_manifest)
assert len(combined_manifest)==6000
assert (combined_manifest.groupby(['dataset','state']).size()==500).all()
combined_manifest.to_csv(
    COMBINED_ROOT/'manifests'/'gemma_four_state_768_manifest.csv',index=False)
display(combined_manifest.groupby(['dataset','state']).size().rename('n').reset_index())


## 8. Unified four-state 768 summary for the manuscript


In [ ]:
COMBINED_CONFIG=dict(CONFIG)
COMBINED_CONFIG.update({
 'protocol_version':'CMC-R1-GEMMA-FOURSTATE-768-COMBINED-2026-09-20',
 'output_root':str(COMBINED_ROOT),
 'states':['llm_only','flow','graph','detector'],
 'expected_packet_states_per_target':4,
})
COMBINED_CONFIG_PATH=Path('/content/cmc_r1_gemma_four_state_768_combined.yaml')
COMBINED_CONFIG_PATH.write_text(
    yaml.safe_dump(COMBINED_CONFIG,sort_keys=False),encoding='utf-8')
(COMBINED_ROOT/'cmc_r1_gemma_four_state_768_combined.yaml').write_text(
    yaml.safe_dump(COMBINED_CONFIG,sort_keys=False),encoding='utf-8')

subprocess.run([
    sys.executable,'scripts/03_recompute_primary_metrics.py',
    '--experiment',str(COMBINED_CONFIG_PATH)
],check=True)

record_metrics=pd.read_csv(COMBINED_ROOT/'results'/'record_metrics_audit_v2.csv')

from cmc87708.evaluation import protected_field_metrics
def exact_field_denominator(packet):
    m=protected_field_metrics(packet,'')
    return sum(1 for k in m if k.endswith('_ok'))

denom=[]
for ds in DATASETS:
    packets=load_jsonl(COMBINED_ROOT/'packets'/f'{ds}.jsonl')
    for state in ['llm_only','flow','graph','detector']:
        sample=next(p for p in packets if p['state']==state)
        denom.append({
            'dataset':ds,'state':state,
            'exact_check_fields':exact_field_denominator(sample)
        })
denom=pd.DataFrame(denom)

summary=(
    record_metrics[record_metrics['method'].eq('proposed_source_gated')]
    .groupby(['dataset','state'],as_index=False)
    .agg(
        n=('edge_id','size'),
        mean_required_field_preservation=('native_required_field_completeness','mean'),
        truncation_rate=('truncated_at_max_tokens','mean'),
        fallback_rate=('fallback_used','mean'),
        mean_generated_tokens=('generated_tokens','mean'),
    )
    .merge(denom,on=['dataset','state'],how='left')
)

state_order=pd.CategoricalDtype(['llm_only','flow','graph','detector'],ordered=True)
dataset_order=pd.CategoricalDtype(DATASETS,ordered=True)
summary['state']=summary['state'].astype(state_order)
summary['dataset']=summary['dataset'].astype(dataset_order)
summary=summary.sort_values(['dataset','state']).reset_index(drop=True)

out_csv=COMBINED_ROOT/'results'/'four_state_768_manuscript_summary.csv'
summary.to_csv(out_csv,index=False)
display(summary)

print('Saved:',out_csv)
print('COMBINED_ROOT:',COMBINED_ROOT)
